In [125]:
import subprocess
import os
import csv
import pandas as pd
# from tqdm import tqdm
# from datetime import datetime
from bayes_opt import BayesianOptimization
from pathlib import Path

In [126]:
# function to print elapsed time
def calculate_elapsed_time(start_time, end_time):
    elapsed_time = end_time - start_time
    total_seconds = elapsed_time.total_seconds()
    seconds = int(total_seconds % 60)
    total_minutes = total_seconds // 60
    minutes = int(total_minutes % 60)
    hours = int(total_minutes // 60)
    
    seconds = str(seconds).zfill(2)
    minutes = str(minutes).zfill(2)
    hours = str(hours).zfill(2)

    return str(hours) + ':' + str(minutes) + ':' + str(seconds)



In [127]:
# VOT config
# alphaWalk,alphaBike,alphaCarDriver,alphaCarPassenger,alphaBus,alphaTrain,betaChangesTransport,betaCostLow,betaCostMed,betaCostHigh,votCommuteWalk,votCommuteBike,votCommuteCar,votCommuteBus,votCommuteTrain,votOtherWalk,votOtherBike,votOtherCar,votOtherBus,votOtherTrain,carCostKm,ptCostKm,ptBaseCost,weightAccessEgress
defaultAlpha_bounds = (-5, 5)
default_beta_bounds = (-0.35, -0.01)
default_bea_cost_bounds = (-0.2,-0.2)

-2,-4,1.2191936792030145,-0.6975272917873561,3,1.7780089143431113


def tuplePlusMinus(val):
        return (val-1, val+1)

pbounds = {'alphaWalk' :tuplePlusMinus(-2) ,
           'alphaBike' : tuplePlusMinus(-4),
           'alphaCarDriver' : tuplePlusMinus(1.2),
           'alphaCarPassenger' : tuplePlusMinus(-0.7),
           'alphaBus' : tuplePlusMinus(3),
           'alphaTrain' : tuplePlusMinus(1.7),
           'betaTimeWalk'  :(-0.0394666667,-0.0394666667) ,
           'betaTimeBike' : (-0.0346333333,-0.0346333333),
           'betaTimeCarDriver': (-0.0347333333,-0.0347333333),
           'betaTimeCarPassenger' : (-0.0347333333,-0.0347333333),
           'betaTimeBus' : (-0.0237333333,-0.0237333333),
           'betaTimeTrain' : (-0.0336,-0.0336),
           'betaCostCarDriver': default_bea_cost_bounds,
           'betaCostCarPassenger' : default_bea_cost_bounds,
           'betaCostBus': default_bea_cost_bounds,
           'betaCostTrain': default_bea_cost_bounds,
           'betaTimeWalkTransport': default_beta_bounds,
           'betaChangesTransport': default_beta_bounds}

# pbounds = {'alphaWalk': defaultAlpha_bounds,
#            'alphaBike': defaultAlpha_bounds,
#            'alphaCarDriver': defaultAlpha_bounds,
#            'alphaCarPassenger': defaultAlpha_bounds,
#            'alphaBus': defaultAlpha_bounds,
#            'alphaTrain': defaultAlpha_bounds,
#            'betaTimeWalk': (-0.04, -0.04),
#            'betaTimeBike': (-0.03, -0.03),
#            'betaTimeCarDriver': (-0.02, -0.02),
#            'betaTimeCarPassenger': (-0.02, -0.02),
#            'betaTimeBus': (-0.02, -0.02),
#            'betaTimeTrain': (-0.02, -0.02),
#            'betaCostCarDriver': (-0.15, -0.15),
#            'betaCostCarPassenger': (-0.15, -0.15),
#            'betaCostBus': (-0.1, -0.1),
#            'betaCostTrain': (-0.1, -0.1),
#            'betaTimeWalkTransport': (-0.03, -0.03),
#            'betaChangesTransport': (-0.3, -0.3)}

# #0.3171403 | -0.457180 | -6.946567 | 2.4361246 | 0.8802023 | 3.0827469 to beat: 879801.5
# pbounds = {'alphaWalk' :(0,3) ,
#     'alphaBike' : (-1,0),
#     'alphaCarDriver' : (-8,-4),
#     'alphaCarPassenger' : (1,4),
#     'alphaBus' : (-1,3),
#     'alphaTrain' : (2,4),
#     'betaTimeWalk' : (-0.04,-0.04),
#     'betaTimeBike' : (-0.03,-0.03),
#     'betaTimeCarDriver': (-0.02,-0.02),
#     'betaTimeCarPassenger' : (-0.02,-0.02),
#     'betaTimeBus' : (-0.02,-0.02),
#     'betaTimeTrain' : (-0.02,-0.02),
#     'betaCostCarDriver': (-0.15,-0.15),
#     'betaCostCarPassenger' : (-0.15,-0.15),
#     'betaCostBus': (-0.1,-0.1),
#     'betaCostTrain': (-0.1,-0.1),
#     'betaTimeWalkTransport': (-0.03,-0.03),
#     'betaChangesTransport': (-0.3,-0.3)}

pbounds

{'alphaWalk': (-3, -1),
 'alphaBike': (-5, -3),
 'alphaCarDriver': (0.19999999999999996, 2.2),
 'alphaCarPassenger': (-1.7, 0.30000000000000004),
 'alphaBus': (2, 4),
 'alphaTrain': (0.7, 2.7),
 'betaTimeWalk': (-0.0394666667, -0.0394666667),
 'betaTimeBike': (-0.0346333333, -0.0346333333),
 'betaTimeCarDriver': (-0.0347333333, -0.0347333333),
 'betaTimeCarPassenger': (-0.0347333333, -0.0347333333),
 'betaTimeBus': (-0.0237333333, -0.0237333333),
 'betaTimeTrain': (-0.0336, -0.0336),
 'betaCostCarDriver': (-0.2, -0.2),
 'betaCostCarPassenger': (-0.2, -0.2),
 'betaCostBus': (-0.2, -0.2),
 'betaCostTrain': (-0.2, -0.2),
 'betaTimeWalkTransport': (-0.35, -0.01),
 'betaChangesTransport': (-0.35, -0.01)}

In [128]:
def writeVotConfigFile(
        alphaWalk,
        alphaBike,
        alphaCarDriver,
        alphaCarPassenger,
        alphaBus,
        alphaTrain,
        betaTimeWalk ,
        betaTimeBike,
        betaTimeCarDriver,
        betaTimeCarPassenger,
        betaTimeBus ,
        betaTimeTrain ,
        betaCostCarDriver,
        betaCostCarPassenger ,
        betaCostBus,
        betaCostTrain,
        betaTimeWalkTransport,
        betaChangesTransport,
        filename) -> int:

    cols = [
        'alphaWalk', 'alphaBike', 'alphaCarDriver', 'alphaCarPassenger', 'alphaBus', 
        'alphaTrain', 'betaTimeWalk' ,
           'betaTimeBike',
           'betaTimeCarDriver',
           'betaTimeCarPassenger',
           'betaTimeBus' ,
           'betaTimeTrain' ,
           'betaCostCarDriver',
           'betaCostCarPassenger' ,
           'betaCostBus',
           'betaCostTrain',
           'betaTimeWalkTransport',
           'betaChangesTransport',
           'carCostKm',
                'ptCostKm',
                'ptBaseCost',
            ]
        
    data = {'alphaWalk' : alphaWalk,
                'alphaBike' :alphaBike,
                'alphaCarDriver': alphaCarDriver,
                'alphaCarPassenger':alphaCarPassenger,
                'alphaBus':alphaBus,
                'alphaTrain' :alphaTrain,
                'betaTimeWalk' : betaTimeWalk,
                'betaTimeBike' : betaTimeBike,
                'betaTimeCarDriver': betaTimeCarDriver,
                'betaTimeCarPassenger' : betaTimeCarPassenger,
                'betaTimeBus' : betaTimeBus,
                'betaTimeTrain' : betaTimeTrain,
                'betaCostCarDriver': betaCostCarDriver,
                'betaCostCarPassenger' : betaCostCarPassenger,
                'betaCostBus': betaCostBus,
                'betaCostTrain': betaCostTrain,
                'betaTimeWalkTransport': betaTimeWalkTransport,
                'betaChangesTransport': betaChangesTransport,
                'carCostKm':0.25,
                'ptCostKm':0.187,
                'ptBaseCost':1.08,}
    if Path(filename).exists():
        config = pd.read_csv(filename)
        new_entry = pd.DataFrame([data], columns=cols)


        #config = pd.concat([config, new_entry], ignore_index=True)

        row_count = len(config) #Length off by one, so new index

        new_entry.to_csv(filename,mode='a',header=(not os.path.exists(filename)) , index = None)

        return row_count
    else:
     
        df = pd.DataFrame([data], columns=cols)


        df.to_csv(filename, index=None)
        return 0
    
    

### Prepare the Java command

In [129]:
def callSimulation(
        alphaWalk,
        alphaBike,
        alphaCarDriver,
        alphaCarPassenger,
        alphaBus,
        alphaTrain,
        betaTimeWalk ,
        betaTimeBike,
        betaTimeCarDriver,
        betaTimeCarPassenger,
        betaTimeBus ,
        betaTimeTrain ,
        betaCostCarDriver,
        betaCostCarPassenger ,
        betaCostBus,
        betaCostTrain,
        betaTimeWalkTransport,
        betaChangesTransport):
    
    filename = "../7-simulation-Sim-2APL/src/main/resources/baseline_parameterset/stt_parameterset_bayesian.csv"
    parameter_set_index = writeVotConfigFile(
        alphaWalk,
        alphaBike,
        alphaCarDriver,
        alphaCarPassenger,
        alphaBus,
        alphaTrain,
        betaTimeWalk ,
        betaTimeBike,
        betaTimeCarDriver,
        betaTimeCarPassenger,
        betaTimeBus ,
        betaTimeTrain ,
        betaCostCarDriver,
        betaCostCarPassenger ,
        betaCostBus,
        betaCostTrain,
        betaTimeWalkTransport,
        betaChangesTransport,
        filename
        )
    os.chdir("C:\\Users\\ed\\Development\\dhzw\\8-sensitivity-analysis")

    df = pd.DataFrame(columns=["CAR_DRIVER", "CAR_PASSENGER", "BIKE", "BUS_TRAM", "TRAIN", "WALK"])

    df.to_csv("output/output_proportions_" + str(parameter_set_index) + ".csv", index=False)

    config = "--config src/main/resources/config_DHZW_stt_demo.toml"
    parameters_path = "--parameter_file src/main/resources/baseline_parameterset/stt_parameterset_bayesian.csv"
    output_path = f'--output_file src/main/resources/baseline_parameterset/output_proportionsCOPY.csv'#../../8-sensitivity-analysis/output/iterations/output_proportions_{parameter_set_index}.csv'
    arg = f'--parameterset_index {parameter_set_index}'
    java_folder_path = '7-simulation-Sim-2APL'
    #print(os.getcwd())
    # set current directory the folder of Sim2APL so I can execute the jar with the correct paths
    if (os.path.basename(os.getcwd()) != java_folder_path):
        new_directory = os.path.join(os.getcwd(), "../", java_folder_path)
        new_directory = new_directory.replace('\\', '/')
        os.chdir(new_directory)
    #print(os.getcwd())

    # parameterset to use
    full_command = f'java -cp target/sim2apl-dhzw-simulation-1.0-SNAPSHOT-jar-with-dependencies.jar main.java.nl.uu.iss.ga.Simulation {config} {output_path} {parameters_path} {arg}'
    #print(full_command)
    try:
        output = subprocess.check_output(
            full_command, stderr=subprocess.STDOUT, universal_newlines=True)
        #print(output)
        return -float(output)
    except subprocess.CalledProcessError as e:
        print(f"Java program exited with non-zero return code: {e.returncode}")
        print(f"Error message: {e.output}")
        exit(1)
    return -999999999999 #an egregiously bad option.

In [130]:
import os
import subprocess


jdk_11_path = r"C:\Program Files\Java\jdk-11"

os.environ["JAVA_HOME"] = jdk_11_path
bin_path = os.path.join(jdk_11_path, "bin")
os.environ["PATH"] = bin_path + os.pathsep + os.environ.get("PATH", "")

try:
    result = subprocess.run(["java", "-version"], capture_output=True, text=True, check=True)
    print("Java Version Output:\n", result.stderr) 
except subprocess.CalledProcessError as e:
    print(f"Error: {e}")
except FileNotFoundError:
    print("Java executable not found in the updated PATH.")

Java Version Output:
 openjdk version "11.0.20.1" 2023-08-24 LTS
OpenJDK Runtime Environment Microsoft-8297088 (build 11.0.20.1+1-LTS)
OpenJDK 64-Bit Server VM Microsoft-8297088 (build 11.0.20.1+1-LTS, mixed mode)



In [131]:
optimizer = BayesianOptimization(
    f=callSimulation,
    pbounds=pbounds,
    random_state=1,
)

In [132]:
optimizer.maximize(
    init_points=40,
    n_iter=40,
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaCo... | betaCo... | betaCo... | betaCo... | betaTi... | betaCh... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -5.126782 | -2.165955 | -3.559351 | 0.2002287 | -1.095334 | 2.2935117 | 0.8846771 | -0.039466 | -0.034633 | -0.034733 | -0.034733 | -0.023733 | -0.0336   | -0.2      | -0.2      | -0.2      | -0.2      | -0.208116 | -0.160045 |
| 2         | -6.327700 | -2.719226 | -4.603797 | 1.8014891 | 0.2365231 | 2.6268483 | 2.0846452 | -0.039466 | -0.034633 | -0.034733 | -0.034733 | -0.023733 | -0.0336   | -0.2      | -0.2      | -0.2      | -0.2      | -0.114761 | -0.242724 |
| 3         | -4.940842 | -1.626

In [133]:
optimizer.max

{'target': np.float64(-3.1524913636626475),
 'params': {'alphaWalk': np.float64(-2.9377775611386787),
  'alphaBike': np.float64(-4.77947050204701),
  'alphaCarDriver': np.float64(1.7924016827063909),
  'alphaCarPassenger': np.float64(0.14752466347254756),
  'alphaBus': np.float64(2.028706059394456),
  'alphaTrain': np.float64(1.8526954137408589),
  'betaTimeWalk': np.float64(-0.0394666667),
  'betaTimeBike': np.float64(-0.0346333333),
  'betaTimeCarDriver': np.float64(-0.0347333333),
  'betaTimeCarPassenger': np.float64(-0.0347333333),
  'betaTimeBus': np.float64(-0.0237333333),
  'betaTimeTrain': np.float64(-0.0336),
  'betaCostCarDriver': np.float64(-0.2),
  'betaCostCarPassenger': np.float64(-0.2),
  'betaCostBus': np.float64(-0.2),
  'betaCostTrain': np.float64(-0.2),
  'betaTimeWalkTransport': np.float64(-0.2154913915808982),
  'betaChangesTransport': np.float64(-0.254783235889478)}}

In [134]:
callSimulation(**optimizer.max['params'])

-3.141526662689793

### Main loop to call the simulations

#20 iterations of sequential. 
{'target': np.float64(-1618722.5288986785), 'params': {'alphaWalk': np.float64(-10.0), 'alphaBike': np.float64(-3.196817588926173), 'alphaCarDriver': np.float64(4.888472151768744), 'alphaCarPassenger': np.float64(10.0), 'alphaBus': np.float64(6.829128765087262), 'alphaTrain': np.float64(10.0), 'betaChangesTransport': np.float64(0.001), 'betaCostLow': np.float64(1.0), 'betaCostMed': np.float64(1.0), 'betaCostHigh': np.float64(1.0), 'weightWalk': np.float64(1.0), 'weightWait': np.float64(3.0), 'weightFeeder': np.float64(1.0)}}

#20 iterations of aggregate.
{'target': np.float64(-1805588.612258473), 'params': {'alphaWalk': np.float64(-10.0), 'alphaBike': np.float64(-5.075847260627779), 'alphaCarDriver': np.float64(3.316409577447148), 'alphaCarPassenger': np.float64(10.0), 'alphaBus': np.float64(-0.3293318074942246), 'alphaTrain': np.float64(10.0), 'betaChangesTransport': np.float64(0.001), 'betaCostLow': np.float64(0.001), 'betaCostMed': np.float64(1.0), 'betaCostHigh': np.float64(1.0), 'weightWalk': np.float64(0.0), 'weightWait': np.float64(2.2692369147311315), 'weightFeeder': np.float64(3.0)}}